## Задание 1. Вывести рекуррентное уравнение для задачи о рюкзаке

**Постановка.** Дан набор из $n$ предметов. Предмет $i$ имеет вес $w_i > 0$ и ценность $v_i > 0$. Вместимость рюкзака $W$.

$$\max \sum_{i=1}^n v_i x_i \quad \text{s.t.}\quad \sum_{i=1}^n w_i x_i \le W,\quad x_i \in \{0, 1\}$$

**Задача.** Ввести $V_k(w)$ — максимальную ценность из первых $k$ предметов при вместимости $w$. Выписать рекуррентное уравнение и граничные условия. Объяснить, почему принцип оптимальности Беллмана выполняется.

In [1]:
import numpy as np
from itertools import combinations


def knapsack_dp_2d(items, W):
    """0/1 knapsack по уравнению V_k(w).

    V_k(w) = V_{k-1}(w),                          w_k > w
           = max(V_{k-1}(w), V_{k-1}(w-w_k)+v_k), w_k <= w

    Returns:
        V   : (n+1, W+1) таблица значений
        opt : V_n(W)
        chosen : список индексов выбранных предметов (0-based)
    """
    n = len(items)
    V = np.zeros((n + 1, W + 1), dtype=int)
    take = np.zeros((n + 1, W + 1), dtype=bool)   # take[k][w] = брать ли предмет k

    for k in range(1, n + 1):
        w_k, v_k = items[k - 1]
        for w in range(W + 1):
            # случай 1: предмет k не влезает
            if w_k > w:
                # TODO
                take[k][w] = False
            # случай 2: предмет k влезает — выбираем максимум
            else:
                without = V[k - 1][w]
                with_k = V[k - 1][w - w_k] + v_k
                if with_k > without:
                    # TODO
                else:
                    # TODO
                    take[k][w] = False

    # восстановление решения: идём с (n, W) назад
    chosen = []
    w = W
    for k in range(n, 0, -1):
        if take[k][w]:
            chosen.append(k - 1)
            w -= items[k - 1][0]
    chosen.reverse()

    return V, int(V[n][W]), chosen

In [ ]:
#TODO пример запуска knapsack_dp_2d

In [2]:
def knapsack_bruteforce(items, W):
    """Перебор всех подмножеств. Только для n <= 15."""
    n = len(items)
    best_val, best_set = 0, []
    for r in range(n + 1):
        for subset in combinations(range(n), r):
            total_w = sum(items[i][0] for i in subset)
            total_v = sum(items[i][1] for i in subset)
            if total_w <= W and total_v > best_val:
                best_val, best_set = total_v, list(subset)
    return best_val, best_set

In [3]:
rng = np.random.default_rng(42)

def test_dp_matches_bruteforce(n_trials=100, max_n=10, max_W=50):
    for trial in range(n_trials):
        n = int(rng.integers(1, max_n + 1))
        W = int(rng.integers(5, max_W + 1))
        items = [(int(rng.integers(1, 15)), int(rng.integers(1, 30)))
                 for _ in range(n)]

        _, val_dp, chosen_dp = knapsack_dp_2d(items, W)
        val_bf, chosen_bf = knapsack_bruteforce(items, W)

        assert val_dp == val_bf, (
            f"trial {trial}: DP={val_dp}, BF={val_bf}, items={items}, W={W}"
        )
        # проверка допустимости восстановленного решения
        total_w = sum(items[i][0] for i in chosen_dp)
        total_v = sum(items[i][1] for i in chosen_dp)
        assert total_w <= W, f"trial {trial}: overweight {total_w} > {W}"
        assert total_v == val_dp, (
            f"trial {trial}: chosen value {total_v} != opt {val_dp}"
        )

    print(f"OK: {n_trials} trials — DP == brute force, "
          f"reconstruction feasible and optimal")

test_dp_matches_bruteforce()

OK: 100 trials — DP == brute force, reconstruction feasible and optimal


**Запишите.** Уравнение и граничные условия: TODO

TODO: Как в функции knapsack_dp_2d восстановить решение, если будет дополнительное условие на количество предметов?

## Задание 2. Найти ошибку в рекуррентном уравнении

**Постановка.** Дано уравнение для рюкзака:

$$V(w) = \max_{i:\ w_i \le w} \big\{ V(w - w_i) + v_i \big\}, \qquad V(0) = 0.$$

Утверждается, что оно решает 0/1 рюкзак.

**Задача.**
1. Проверить корректность.
2. Найти контрпример, где уравнение даёт неверный ответ.
3. Объяснить, почему для 0/1 нужен дополнительный индекс $k$.
4. Выписать исправленное уравнение.

**Что решает.** Это уравнение для **неограниченного рюкзака** (unbounded knapsack). Оно не отслеживает, какие предметы уже использованы, и может взять один и тот же предмет несколько раз.

**Контрпример.** $n = 2$, $W = 10$:

| Предмет | $w_i$ | $v_i$ |
|---|---:|---:|
| 1 | 6 | 10 |
| 2 | 5 | 9 |

- **Истинный оптимум 0/1:** набор $\{1\}$ весит 6, даёт 10. Набор $\{1,2\}$ весит 11 > 10. Оптимум = **10**.
- **Что даёт уравнение:** $V(5) = 9$, $V(6) = 10$, $V(10) = \max\{V(4)+10,\ V(5)+9\} = \max\{10,\ 18\} = 18$.
- Уравнение даёт **18**: оно «взяло» предмет 2 дважды (вес $5+5=10$, ценность $9+9=18$). Для 0/1 — неверно.

**Исправленное уравнение.**

$$V_k(w) =
\begin{cases}
V_{k-1}(w), & w_k > w, \\
\max\{ V_{k-1}(w),\ V_{k-1}(w - w_k) + v_k \}, & w_k \le w.
\end{cases}$$

Ключевое: в $V_{k-1}(w - w_k)$ используется $k-1$, а не $k$ — предмет $k$ уже не может быть взят повторно.

## Задание 3. Три способа решить 0/1 рюкзак

Постановка. Реализовать три подхода:

Наивная рекурсия без мемоизации $O(?)$. # TODO

Top-down с мемоизацией — $O(?)$ состояний, каждое считается один раз. # TODO

Bottom-up DP с 1D-массивом — $O(?)$ времени, $O(?)$ памяти, обратный проход по $w$. # TODO

Проверить. Все три дают одинаковый ответ на 20 случайных экземплярах при $n \leq 12$. Замерить время на 
$n=25$, $W=1000$.

In [17]:
# Наивная рекурсия
from functools import lru_cache


def knapsack_naive(items, W):
    """0/1 рюкзак, наивная рекурсия без мемоизации. O(?).

    items: список пар (w_i, v_i)
    W: вместимость
    Возвращает максимальную ценность.
    """
    n = len(items)

    def rec(k, cap):
        # база: нет предметов или нет места
        if k == 0 or cap == 0:
            return 0
        w, v = items[k - 1]
        # предмет k не влезает — пропускаем
        if w > cap:
            return # TODO
        return # TODO

    return rec(n, W)

In [19]:
#top-down с меморизациейdef knapsack_memo(items, W):
def knapsack_memo(items, W):
    """0/1 рюкзак, top-down с мемоизацией. O(?) состояний."""
    n = len(items)

    @lru_cache(maxsize=None)
    def rec(k, cap):
        if k == 0 or cap == 0:
            return 0
        w, v = items[k - 1]
        if w > cap:
            return # TODO
        return # TODO

    result = rec(n, W)
    rec.cache_clear()   # чтобы не держать память между вызовами
    return result


In [20]:
# bottom-up DP
def knapsack_dp_1d(items, W):
    """0/1 рюкзак, bottom-up 1D DP. O(?) время, O(?) память.
    """
    dp = [0] * (W + 1)
    for w, v in items:
        for cap in range(W, w - 1, -1):    
            if dp[cap - w] + v > dp[cap]:
                # TODO
    return dp[W]

In [21]:
import numpy as np

def test_three_methods_agree(n_trials=100, max_n=12, max_W=60, seed=42):
    """Проверяет, что naive == memo == dp_1d на случайных экземплярах."""
    rng = np.random.default_rng(seed)
    for trial in range(n_trials):
        n = int(rng.integers(1, max_n + 1))
        W = int(rng.integers(5, max_W + 1))
        items = [(int(rng.integers(1, 15)), int(rng.integers(1, 30)))
                 for _ in range(n)]

        a = knapsack_naive(items, W)
        b = knapsack_memo(items, W)
        c = knapsack_dp_1d(items, W)

        assert a == b == c, (
            f"trial {trial}: naive={a}, memo={b}, dp_1d={c}\n"
            f"  items={items}, W={W}"
        )

    print(f"OK: all {n_trials} trials agree (naive == memo == dp_1d)")


test_three_methods_agree()

OK: all 100 trials agree (naive == memo == dp_1d)


In [10]:
# TODO: сравнение на 20 лучайных экзмеплярах, проверка времени

## Задание 7. Распределение ресурсов: DP и жадный

Постановка. $n=4$ агента, бюджет $B=10$. Агент $i$ получает $x_i \in {0,…,B}$ и приносит $f_i(x_i)$.

$$\max_{x_1+⋯+x_n\leq B} \sum_{i=1}^n f_i(x_i).$$
Два случая.

A — вогнутые: $f_i(x)=a_i\sqrt{x}, a=(3,2,4,1)$.
B — невогнутые с плато: 
- $f_1$: 0→0, 1→5, 2→5, 3→5,…
- $f_2$: 0→0, 1→4, 2→8, 3→8,…
- $f_3$: 0→0, 1→6, 2→6, 3→6,…
- $f_4$:0→0, 1→3, 2→7, 3→7,…

Задача. Для обоих случаев решить DP и жадным по маргинальной полезности 
$\Delta f_i(x)=f_i(x)−f_i(x−1)$. Сравнить. Для невогнутых привести контрпример, где жадный строго хуже DP.

In [11]:
# TODO